# Automotive Sentiment Analysis — DistilBERT Fine-Tuning

Fine-tuning `distilbert-base-uncased` on an expanded, 27K-row automotive review dataset (3-class: Negative / Neutral / Positive) as a follow-up to an earlier TF-IDF + Logistic Regression baseline (macro F1 0.67).

**Goal:** see whether a transformer's contextual understanding can improve on the Neutral-class recall ceiling (~48%) that classical ML couldn't break.

## 1. Environment setup
Check GPU availability and install the Hugging Face ecosystem.

In [3]:
!nvidia-smi

Tue Aug 25 13:43:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install transformers datasets torch

## 2. Load data
Mount Google Drive and load the expanded, merged dataset (original 4,716 rows + ~22.5K new rows sourced from the Edmunds Kaggle dataset across 49 brands).

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
merged = pd.read_csv('/content/drive/MyDrive/merged_27k_dataset.csv')
print(merged.shape)
print(merged['Sentiment'].value_counts())

(27155, 2)
Sentiment
Positive    9982
Neutral     9529
Negative    7644
Name: count, dtype: int64


## 3. Train / validation / test split
70/15/15 stratified split to preserve class proportions across all three sets.

In [7]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(merged, test_size=0.3, stratify=merged['Sentiment'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['Sentiment'], random_state=42)

print(train_df.shape, val_df.shape, test_df.shape)

(19008, 2) (4073, 2) (4074, 2)


## 4. Label encoding
Convert text labels (Negative/Neutral/Positive) into integers (0/1/2) for the model.

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['Sentiment'])
val_df['label'] = le.transform(val_df['Sentiment'])
test_df['label'] = le.transform(test_df['Sentiment'])
print(le.classes_)

['Negative' 'Neutral' 'Positive']


## 5. Tokenization
Load the DistilBERT tokenizer and apply subword tokenization (with padding + truncation) to all three splits.

In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples['Review'], padding=True, truncation=True, max_length=128)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[['Review', 'label']]).map(tokenize_function, batched=True)
val_ds = Dataset.from_pandas(val_df[['Review', 'label']]).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df[['Review', 'label']]).map(tokenize_function, batched=True)

Map:   0%|          | 0/19008 [00:00<?, ? examples/s]

Map:   0%|          | 0/4073 [00:00<?, ? examples/s]

Map:   0%|          | 0/4074 [00:00<?, ? examples/s]

## 6. Model + evaluation metric
Load pretrained DistilBERT with a fresh 3-class classification head, and define macro F1 / accuracy as the evaluation metric.

In [10]:
from transformers import AutoModelForSequenceClassification
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="macro")
    acc = accuracy_score(labels, predictions)
    return {"f1": f1, "accuracy": acc}

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 7. Fine-tuning
Train for 3 epochs with a small learning rate (2e-5) to preserve pretrained knowledge while adapting to the sentiment task. Best checkpoint (by validation F1) is kept automatically.

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_bigdata",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.710849,0.625101,0.708877,0.710287
2,0.569817,0.610535,0.719743,0.722563
3,0.507653,0.621206,0.724586,0.728456


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1782, training_loss=0.5740052553941104, metrics={'train_runtime': 676.5424, 'train_samples_per_second': 84.287, 'train_steps_per_second': 2.634, 'total_flos': 1888488913158144.0, 'train_loss': 0.5740052553941104, 'epoch': 3.0})

## 8. Final evaluation on the test set
Honest, held-out evaluation — this is what actually gets compared against the TF-IDF baseline.

In [12]:
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(test_ds['label'], preds, target_names=le.classes_))

              precision    recall  f1-score   support

    Negative       0.68      0.69      0.69      1146
     Neutral       0.62      0.61      0.61      1430
    Positive       0.83      0.85      0.84      1498

    accuracy                           0.72      4074
   macro avg       0.71      0.71      0.71      4074
weighted avg       0.72      0.72      0.72      4074



## Results

| Model | Macro F1 | Accuracy | Neutral Recall |
|---|---|---|---|
| TF-IDF + bigram + LogReg (baseline, 4,716 rows) | 0.67 | 69% | 48% |
| DistilBERT (this notebook, 27,155 rows) | **0.71** | **72%** | **61%** |

First version to clearly beat the TF-IDF baseline on every metric, including a 13-point jump in Neutral recall.

**Caveat:** this isn't a perfectly clean same-data comparison — the baseline was measured on the original 4,716-row test split, this model on the expanded 27,155-row test split. Both more data and a better architecture likely contributed to the gain.